# Step 5: Memory Management & Tuning

## Learning Objectives
1. Spark memory structure (Driver / Executor)
2. Understanding the Unified Memory Manager
3. Storage vs Execution memory
4. Understanding and monitoring Spill (disk spill)
5. Observing GC (Garbage Collection)
6. Key tuning parameters
7. OOM debugging patterns

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
import time
import os

spark = SparkSession.builder \
    .appName("Step5-Memory-Tuning") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.adaptive.enabled", "false") \
    .config("spark.sql.shuffle.partitions", "20") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/data/warehouse") \
    .getOrCreate()

sc = spark.sparkContext
print(f"Driver memory:   {spark.conf.get('spark.driver.memory')}")
print(f"Executor memory: {spark.conf.get('spark.executor.memory')}")
print(f"Executor cores:  {spark.conf.get('spark.executor.cores')}")
print(f"✅ Spark UI: http://localhost:4040")

Driver memory:   1g
Executor memory: 1g
Executor cores:  1
✅ Spark UI: http://localhost:4040


---
## 1. Spark Memory Structure

### Driver Memory
```
┌──────────────────────────────────────┐
│           Driver JVM Heap            │
│         (spark.driver.memory)        │
├──────────────────────────────────────┤
│  SparkContext, DAG Scheduler         │
│  Broadcast variables (original)     │
│  collect() results                   │
│  Accumulator values                  │
└──────────────────────────────────────┘
   + Off-heap: spark.driver.memoryOverhead (default 10%)
```

### Executor Memory
```
┌──────────────────────────────────────────────┐
│              Executor JVM Heap                │
│           (spark.executor.memory)             │
├──────────────────────────────────────────────┤
│                                              │
│  Reserved Memory (300MB fixed)               │
│                                              │
├──────────────────────────────────────────────┤
│                                              │
│  Unified Memory                              │
│  (spark.memory.fraction = 0.6)               │
│  ┌────────────────┬─────────────────┐        │
│  │  Storage       │  Execution      │        │
│  │  Memory        │  Memory         │        │
│  │  (cache,       │  (Shuffle,      │        │
│  │   broadcast)   │   Sort, Join,   │        │
│  │                │   Aggregation)  │        │
│  │  ◄──── fluid boundary ────►      │        │
│  │  (storageFraction = 0.5)         │        │
│  └────────────────┴─────────────────┘        │
│                                              │
├──────────────────────────────────────────────┤
│                                              │
│  User Memory (remaining 40%)                 │
│  (UDF, data structures, Python worker, etc.) │
│                                              │
└──────────────────────────────────────────────┘
   + Off-heap: spark.executor.memoryOverhead (default 10%)
   + PySpark: spark.executor.pyspark.memory (Python-specific)
```

In [2]:
# Check current memory settings
configs = [
    ("spark.executor.memory",         "Executor Heap"),
    ("spark.executor.memoryOverhead",  "Executor Off-heap overhead"),
    ("spark.executor.cores",           "Executor core count"),
    ("spark.driver.memory",            "Driver Heap"),
    ("spark.memory.fraction",          "Unified Memory ratio"),
    ("spark.memory.storageFraction",   "Storage Memory initial ratio"),
    ("spark.memory.offHeap.enabled",   "Off-heap enabled"),
    ("spark.memory.offHeap.size",      "Off-heap size"),
    ("spark.sql.shuffle.partitions",   "Shuffle partition count"),
]

print(f"{'Config':<45} {'Value':>15}  Description")
print("=" * 90)
for key, desc in configs:
    try:
        val = spark.conf.get(key)
    except:
        val = "(default)"
    print(f"{key:<45} {val:>15}  {desc}")

Config                                                  Value  Description
spark.executor.memory                                      1g  Executor Heap
spark.executor.memoryOverhead                       (default)  Executor Off-heap overhead
spark.executor.cores                                        1  Executor core count
spark.driver.memory                                        1g  Driver Heap
spark.memory.fraction                               (default)  Unified Memory ratio
spark.memory.storageFraction                        (default)  Storage Memory initial ratio
spark.memory.offHeap.enabled                        (default)  Off-heap enabled
spark.memory.offHeap.size                           (default)  Off-heap size
spark.sql.shuffle.partitions                               20  Shuffle partition count


In [3]:
# Memory calculation example
executor_memory_mb = 1024  # 1g
reserved_mb = 300
memory_fraction = 0.6
storage_fraction = 0.5

usable = executor_memory_mb - reserved_mb
unified = usable * memory_fraction
storage_init = unified * storage_fraction
execution_init = unified * (1 - storage_fraction)
user_memory = usable * (1 - memory_fraction)

print(f"""
=== Executor Memory Breakdown (1GB basis) ===

  Total Heap:        {executor_memory_mb:>6} MB
  - Reserved:        {reserved_mb:>6} MB  (fixed)
  = Usable:          {usable:>6} MB
  
  Unified Memory:    {unified:>6.0f} MB  ({memory_fraction*100:.0f}%)
    ├─ Storage:      {storage_init:>6.0f} MB  (cache, broadcast)
    └─ Execution:    {execution_init:>6.0f} MB  (shuffle, sort, join)
  
  User Memory:       {user_memory:>6.0f} MB  ({(1-memory_fraction)*100:.0f}%)
                                   (UDF, Python objects, etc.)

💡 Unified Memory has a fluid boundary:
   - When Execution runs low, it borrows from Storage (can evict)
   - When Storage runs low, it borrows from Execution (empty space only)
   - Execution can evict Storage, but not vice versa
""")


=== Executor Memory Breakdown (1GB basis) ===

  Total Heap:          1024 MB
  - Reserved:           300 MB  (fixed)
  = Usable:             724 MB
  
  Unified Memory:       434 MB  (60%)
    ├─ Storage:         217 MB  (cache, broadcast)
    └─ Execution:       217 MB  (shuffle, sort, join)
  
  User Memory:          290 MB  (40%)
                                   (UDF, Python objects, etc.)

💡 Unified Memory has a fluid boundary:
   - When Execution runs low, it borrows from Storage (can evict)
   - When Storage runs low, it borrows from Execution (empty space only)
   - Execution can evict Storage, but not vice versa



In [4]:
random.seed(42)

# Generate data
data = [
    (i, f"name_{i}", random.choice(["A","B","C","D","E"]), 
     random.random() * 100000, "x" * random.randint(10, 200))
    for i in range(500_000)
]
df = spark.createDataFrame(data, ["id", "name", "group", "value", "payload"])

print(f"Row count:  {df.count():,}")
print(f"Partitions: {df.rdd.getNumPartitions()}")

Row count:  500,000
Partitions: 4


In [5]:
from pyspark import StorageLevel

# StorageLevels available in PySpark 3.5
levels = {
    "MEMORY_ONLY":          StorageLevel.MEMORY_ONLY,
    "MEMORY_ONLY_2":        StorageLevel.MEMORY_ONLY_2,          # replicated 2x
    "MEMORY_AND_DISK":      StorageLevel.MEMORY_AND_DISK,
    "MEMORY_AND_DISK_DESER": StorageLevel.MEMORY_AND_DISK_DESER, # deserialized
    "DISK_ONLY":            StorageLevel.DISK_ONLY,
}

print(f"{'StorageLevel':<25} {'Time':>8}")
print("=" * 40)

for name, level in levels.items():
    df.unpersist()
    df.persist(level)
    
    start = time.time()
    df.count()
    elapsed = time.time() - start
    
    print(f"{name:<25} {elapsed:>7.3f}s")

df.unpersist()

print("""
💡 PySpark 3.5 StorageLevel:

  MEMORY_ONLY           → serialized, memory only (default)
  MEMORY_AND_DISK       → serialized, spills to disk when memory is full
  MEMORY_AND_DISK_DESER → deserialized (Java objects, faster but uses more memory)
  DISK_ONLY             → disk only
  _2 suffix             → replicated across 2 nodes (for fault tolerance)

  Check cache size in Spark UI > Storage tab.
""")


StorageLevel                  Time
MEMORY_ONLY                 0.892s
MEMORY_ONLY_2               0.455s
MEMORY_AND_DISK             0.416s
MEMORY_AND_DISK_DESER       0.332s
DISK_ONLY                   0.343s

💡 PySpark 3.5 StorageLevel:

  MEMORY_ONLY           → serialized, memory only (default)
  MEMORY_AND_DISK       → serialized, spills to disk when memory is full
  MEMORY_AND_DISK_DESER → deserialized (Java objects, faster but uses more memory)
  DISK_ONLY             → disk only
  _2 suffix             → replicated across 2 nodes (for fault tolerance)

  Check cache size in Spark UI > Storage tab.



In [6]:
# Measure the effect of caching
df.cache()
df.count()  # warm up

# Repeated operations with cache
start = time.time()
for _ in range(3):
    df.filter(F.col("value") > 50000).groupBy("group").agg(F.avg("value")).collect()
cached_time = time.time() - start

df.unpersist()

# Repeated operations without cache
start = time.time()
for _ in range(3):
    df.filter(F.col("value") > 50000).groupBy("group").agg(F.avg("value")).collect()
uncached_time = time.time() - start

print(f"With cache (3 runs): {cached_time:.3f}s")
print(f"Without cache (3 runs): {uncached_time:.3f}s")
print(f"\n💡 Use cache() on DataFrames you access repeatedly to avoid recomputation.")
print(f"   Always call unpersist() when done!")

With cache (3 runs): 1.277s
Without cache (3 runs): 1.048s

💡 Use cache() on DataFrames you access repeatedly to avoid recomputation.
   Always call unpersist() when done!


---
## 3. Execution Memory & Spill

When Execution Memory is insufficient, data **spills** to disk.

```
Execution Memory sufficient   Execution Memory insufficient
┌──────────┐                  ┌──────────┐
│  memory  │                  │  memory  │ ← full
│  Sort    │                  │  Sort    │
│  Buffer  │                  │  Buffer  │──→ temp files on disk
│          │                  │          │    (Spill)
└──────────┘                  └──────────┘
    Fast!                         Slow (disk I/O)
```

In [7]:
import json, urllib.request

# ~1GB of sortable data, generated natively in the JVM
big_df = (spark.range(2_000_000)
          .withColumn("value", F.rand())
          .withColumn("payload", F.expr("repeat('x', 480)")))

def total_spill_mb():
    """Read cumulative spill (memory & disk) across all stages from the Spark REST API."""
    url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/stages"
    stages = json.load(urllib.request.urlopen(url))
    mem = sum(s.get("memoryBytesSpilled", 0) for s in stages) / 1024 / 1024
    disk = sum(s.get("diskBytesSpilled", 0) for s in stages) / 1024 / 1024
    return mem, disk

# IMPORTANT: orderBy(...).count() does NOT sort — Catalyst prunes the Sort because count()
# does not need order. So we must force the sort with a sink that consumes every ordered row.
print("orderBy().count() keeps a Sort? ",
      "Sort" in big_df.orderBy("value").groupBy().count()._jdf.queryExecution().executedPlan().toString())
print("orderBy() alone keeps a Sort?   ",
      "Sort" in big_df.orderBy("value")._jdf.queryExecution().executedPlan().toString())

def sorted_run(nparts):
    spark.conf.set("spark.sql.shuffle.partitions", str(nparts))
    m0, d0 = total_spill_mb()
    t = time.time()
    big_df.orderBy("value").write.format("noop").mode("overwrite").save()  # forces the global sort
    elapsed = time.time() - t
    m1, d1 = total_spill_mb()
    return elapsed, m1 - m0, d1 - d0

t2,   mem2,   disk2   = sorted_run(2)     # ~1GB funneled into 2 reduce partitions → spill
t200, mem200, disk200 = sorted_run(200)   # ~5MB per partition → no spill
spark.conf.set("spark.sql.shuffle.partitions", "20")

print(f"\n  2 partitions  → {t2:6.2f}s | spill(memory) {mem2:6.0f} MB | spill(disk) {disk2:5.0f} MB")
print(f"200 partitions  → {t200:6.2f}s | spill(memory) {mem200:6.0f} MB | spill(disk) {disk200:5.0f} MB")

print("""
🔍 Spill in the Spark UI: Stages tab → a Stage → Summary Metrics → 'Spill (memory)' / 'Spill (disk)'.
   'Spill (memory)' is the in-memory (uncompressed) size evicted; 'Spill (disk)' is the compressed
   size actually written. With 2 partitions, ~1GB of sorted data cannot fit one task's execution
   memory, so it spills; with 200 partitions each task sorts a few MB and nothing spills.

   NOTE: the orderBy().count() check above keeps NO Sort — Catalyst drops it because count()
   doesn't need order. To observe a real sort/spill you must CONSUME the ordered rows (here a
   noop write sink). When Spill appears:
     1. Increase partition count (less data per task)   2. Increase executor memory
     3. Drop unnecessary columns early                  4. Fix data skew
""")

orderBy().count() keeps a Sort?  False
orderBy() alone keeps a Sort?    True

  2 partitions  →   1.27s | spill(memory)    800 MB | spill(disk)    40 MB
200 partitions  →   1.61s | spill(memory)      0 MB | spill(disk)     0 MB

🔍 Spill in the Spark UI: Stages tab → a Stage → Summary Metrics → 'Spill (memory)' / 'Spill (disk)'.
   'Spill (memory)' is the in-memory (uncompressed) size evicted; 'Spill (disk)' is the compressed
   size actually written. With 2 partitions, ~1GB of sorted data cannot fit one task's execution
   memory, so it spills; with 200 partitions each task sorts a few MB and nothing spills.

   NOTE: the orderBy().count() check above keeps NO Sort — Catalyst drops it because count()
   doesn't need order. To observe a real sort/spill you must CONSUME the ordered rows (here a
   noop write sink). When Spill appears:
     1. Increase partition count (less data per task)   2. Increase executor memory
     3. Drop unnecessary columns early                  4. Fix data ske

---
## 4. Observing GC (Garbage Collection)

In [8]:
# Check GC-related settings
print("=== Key GC Concepts ===")
print(f"""
JVM Heap structure:
┌─────────────────────────────────────┐
│  Young Generation                   │
│  ├─ Eden Space (new objects)        │
│  ├─ Survivor 0                      │
│  └─ Survivor 1                      │
│     → Minor GC (fast, frequent)     │
├─────────────────────────────────────┤
│  Old Generation                     │
│  (long-lived objects)               │
│     → Major/Full GC (slow, rare)    │
└─────────────────────────────────────┘

When GC becomes a problem in Spark:
  1. Cached data fills Old Gen → frequent Full GC
  2. UDF creates many objects → Young Gen pressure
  3. collect() pulls large data to the Driver

GC tuning options (spark.executor.extraJavaOptions):
  -XX:+UseG1GC                            → use G1 GC (recommended)
  -XX:InitiatingHeapOccupancyPercent=35   → GC trigger threshold
  -XX:+PrintGCDetails -XX:+PrintGCTimeStamps  → GC logging
""")

=== Key GC Concepts ===

JVM Heap structure:
┌─────────────────────────────────────┐
│  Young Generation                   │
│  ├─ Eden Space (new objects)        │
│  ├─ Survivor 0                      │
│  └─ Survivor 1                      │
│     → Minor GC (fast, frequent)     │
├─────────────────────────────────────┤
│  Old Generation                     │
│  (long-lived objects)               │
│     → Major/Full GC (slow, rare)    │
└─────────────────────────────────────┘

When GC becomes a problem in Spark:
  1. Cached data fills Old Gen → frequent Full GC
  2. UDF creates many objects → Young Gen pressure
  3. collect() pulls large data to the Driver

GC tuning options (spark.executor.extraJavaOptions):
  -XX:+UseG1GC                            → use G1 GC (recommended)
  -XX:InitiatingHeapOccupancyPercent=35   → GC trigger threshold
  -XX:+PrintGCDetails -XX:+PrintGCTimeStamps  → GC logging



In [9]:
# Experiment to observe GC pressure
# Repeatedly create and release large caches to trigger GC

print("Starting GC pressure experiment...")
print("(Observe GC Time in Spark UI > Executors tab)\n")

times = []
for i in range(5):
    temp_df = spark.range(500_000).withColumn("data", F.rand())
    temp_df.cache()
    
    start = time.time()
    temp_df.groupBy((F.col("id") % 100).alias("grp")).agg(F.sum("data")).count()
    elapsed = time.time() - start
    times.append(elapsed)
    
    temp_df.unpersist()
    print(f"  Run {i+1}: {elapsed:.3f}s")

print(f"\nAverage: {sum(times)/len(times):.3f}s")
print(f"""
🔍 Check in Spark UI > Executors tab:
   - GC Time: total time spent on GC for that executor
   - If GC Time is > 10% of Task Time, tuning is needed
   
   Solutions:
   1. Use MEMORY_AND_DISK_DESER → fewer objects
   2. Release unnecessary cache (unpersist)
   3. Increase executor memory
   4. Use G1 GC + adjust threshold
""")

Starting GC pressure experiment...
(Observe GC Time in Spark UI > Executors tab)

  Run 1: 0.350s
  Run 2: 0.214s
  Run 3: 0.161s
  Run 4: 0.142s
  Run 5: 0.153s

Average: 0.204s

🔍 Check in Spark UI > Executors tab:
   - GC Time: total time spent on GC for that executor
   - If GC Time is > 10% of Task Time, tuning is needed
   
   Solutions:
   1. Use MEMORY_AND_DISK_DESER → fewer objects
   2. Release unnecessary cache (unpersist)
   3. Increase executor memory
   4. Use G1 GC + adjust threshold



---
## 5. Key Tuning Parameters

In [10]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║             Key Spark Memory Tuning Parameters                  ║
╠══════════════════════════════════════════════════════════════════╣

▸ Executor Memory
  spark.executor.memory          = 1g ~ 8g  (typically 4~8g)
  spark.executor.memoryOverhead  = max(384MB, 0.1 * memory)
  spark.executor.pyspark.memory  = 0  (set when using PySpark UDFs)

▸ Executor count / cores
  spark.executor.instances       = prefer dynamic allocation
  spark.executor.cores           = 2 ~ 5  (> 5 causes GC pressure)
  
▸ Driver Memory
  spark.driver.memory            = 1g ~ 4g
  spark.driver.maxResultSize     = 1g  (limit collect result size)

▸ Memory Manager
  spark.memory.fraction          = 0.6  (Unified Memory ratio)
  spark.memory.storageFraction   = 0.5  (Storage initial ratio)

▸ Shuffle
  spark.sql.shuffle.partitions   = 200  (adjust to data size)
  spark.shuffle.spill.compress   = true

▸ Dynamic Allocation
  spark.dynamicAllocation.enabled      = true
  spark.dynamicAllocation.minExecutors = 1
  spark.dynamicAllocation.maxExecutors = 100

╚══════════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════════╗
║             Key Spark Memory Tuning Parameters                  ║
╠══════════════════════════════════════════════════════════════════╣

▸ Executor Memory
  spark.executor.memory          = 1g ~ 8g  (typically 4~8g)
  spark.executor.memoryOverhead  = max(384MB, 0.1 * memory)
  spark.executor.pyspark.memory  = 0  (set when using PySpark UDFs)

▸ Executor count / cores
  spark.executor.instances       = prefer dynamic allocation
  spark.executor.cores           = 2 ~ 5  (> 5 causes GC pressure)
  
▸ Driver Memory
  spark.driver.memory            = 1g ~ 4g
  spark.driver.maxResultSize     = 1g  (limit collect result size)

▸ Memory Manager
  spark.memory.fraction          = 0.6  (Unified Memory ratio)
  spark.memory.storageFraction   = 0.5  (Storage initial ratio)

▸ Shuffle
  spark.sql.shuffle.partitions   = 200  (adjust to data size)
  spark.shuffle.spill.compress   = true

▸ Dynamic Allocation
  spark.dynamicAllocat

In [11]:
# Guide for executor count × core count combinations
print("""
=== Executor Sizing Example ===

Cluster: 10 nodes, 64GB RAM per node, 16 cores per node

❌ Fat Executor (anti-pattern)
   spark.executor.memory = 60g
   spark.executor.cores  = 15
   → Problem: GC managing a huge heap, too many concurrent HDFS writes

❌ Tiny Executor (anti-pattern)
   spark.executor.memory = 1g
   spark.executor.cores  = 1
   → Problem: too many broadcast copies, high JVM overhead ratio

✅ Balanced (recommended)
   spark.executor.memory = 8g ~ 12g
   spark.executor.cores  = 4 ~ 5
   instances = (16 cores / 5 cores) * 10 nodes - 1(AM) ≈ 29
   
   Calculation:
   - Executors per node: 16 / 5 = 3 (1 core reserved for OS)
   - Memory per executor: (64 - 2 OS) / 3 = ~20GB
     → including overhead: 20 * 0.9 ≈ 18GB heap + 2GB overhead
   - Total: 3 * 10 - 1 (YARN AM) = 29 executors
""")


=== Executor Sizing Example ===

Cluster: 10 nodes, 64GB RAM per node, 16 cores per node

❌ Fat Executor (anti-pattern)
   spark.executor.memory = 60g
   spark.executor.cores  = 15
   → Problem: GC managing a huge heap, too many concurrent HDFS writes

❌ Tiny Executor (anti-pattern)
   spark.executor.memory = 1g
   spark.executor.cores  = 1
   → Problem: too many broadcast copies, high JVM overhead ratio

✅ Balanced (recommended)
   spark.executor.memory = 8g ~ 12g
   spark.executor.cores  = 4 ~ 5
   instances = (16 cores / 5 cores) * 10 nodes - 1(AM) ≈ 29
   
   Calculation:
   - Executors per node: 16 / 5 = 3 (1 core reserved for OS)
   - Memory per executor: (64 - 2 OS) / 3 = ~20GB
     → including overhead: 20 * 0.9 ≈ 18GB heap + 2GB overhead
   - Total: 3 * 10 - 1 (YARN AM) = 29 executors



---
## 6. OOM Debugging Patterns

In [12]:
print("""
=== OOM (Out of Memory) Response by Location ===

1️⃣ Driver OOM
   Symptom: "java.lang.OutOfMemoryError: Java heap space" in Driver
   Causes:
     - Pulling large data to Driver via collect() / toPandas()
     - Large broadcast variables
     - Too many partition metadata entries
   Solutions:
     ✅ Increase spark.driver.memory
     ✅ Use take(n), show(n) instead of collect()
     ✅ Limit with spark.driver.maxResultSize

2️⃣ Executor OOM (Heap)
   Symptom: "java.lang.OutOfMemoryError" in Executor / Task failed
   Causes:
     - Partition too large for available memory
     - Data skew (data concentrated in one partition)
     - UDF creates many objects
   Solutions:
     ✅ Increase spark.executor.memory
     ✅ Increase shuffle partition count
     ✅ Data skew → Salting / AQE
     ✅ Drop unnecessary columns early (select first)

3️⃣ Executor OOM (Off-heap / Container killed)
   Symptom: "Container killed by YARN for exceeding memory limits"
   Causes:
     - Python worker memory (PySpark UDF)
     - Off-heap memory usage
     - Native library memory leak
   Solutions:
     ✅ Increase spark.executor.memoryOverhead
     ✅ Set spark.executor.pyspark.memory
     ✅ Convert Python UDF → Pandas UDF
""")


=== OOM (Out of Memory) Response by Location ===

1️⃣ Driver OOM
   Symptom: "java.lang.OutOfMemoryError: Java heap space" in Driver
   Causes:
     - Pulling large data to Driver via collect() / toPandas()
     - Large broadcast variables
     - Too many partition metadata entries
   Solutions:
     ✅ Increase spark.driver.memory
     ✅ Use take(n), show(n) instead of collect()
     ✅ Limit with spark.driver.maxResultSize

2️⃣ Executor OOM (Heap)
   Symptom: "java.lang.OutOfMemoryError" in Executor / Task failed
   Causes:
     - Partition too large for available memory
     - Data skew (data concentrated in one partition)
     - UDF creates many objects
   Solutions:
     ✅ Increase spark.executor.memory
     ✅ Increase shuffle partition count
     ✅ Data skew → Salting / AQE
     ✅ Drop unnecessary columns early (select first)

3️⃣ Executor OOM (Off-heap / Container killed)
   Symptom: "Container killed by YARN for exceeding memory limits"
   Causes:
     - Python worker memory (Py

In [13]:
# Driver OOM simulation (note: does NOT actually trigger OOM)

# ❌ Dangerous pattern
print("❌ Driver OOM risk patterns:")
print()
print("  # 1. Large collect")
print("  all_data = huge_df.collect()  # millions of rows to Driver")
print()
print("  # 2. Large toPandas")
print("  pdf = huge_df.toPandas()  # everything into Driver memory")
print()
print("  # 3. Large broadcast")
print("  large_lookup = spark.read.parquet('1gb_file')")
print("  df.join(F.broadcast(large_lookup), ...)  # 1GB to every executor")
print()

# ✅ Safe patterns
print("\n✅ Safe alternatives:")
print()
print("  # 1. Fetch a limited number of rows")
print("  sample = huge_df.take(1000)")
print("  huge_df.show(20)")
print()
print("  # 2. toPandas only on aggregated (small) results")
print("  summary = huge_df.groupBy('col').count().toPandas()")
print()
print("  # 3. Write to file, then read separately")
print("  huge_df.write.parquet('output/')")
print("  # analyze with a separate tool")

❌ Driver OOM risk patterns:

  # 1. Large collect
  all_data = huge_df.collect()  # millions of rows to Driver

  # 2. Large toPandas
  pdf = huge_df.toPandas()  # everything into Driver memory

  # 3. Large broadcast
  large_lookup = spark.read.parquet('1gb_file')
  df.join(F.broadcast(large_lookup), ...)  # 1GB to every executor


✅ Safe alternatives:

  # 1. Fetch a limited number of rows
  sample = huge_df.take(1000)
  huge_df.show(20)

  # 2. toPandas only on aggregated (small) results
  summary = huge_df.groupBy('col').count().toPandas()

  # 3. Write to file, then read separately
  huge_df.write.parquet('output/')
  # analyze with a separate tool


In [14]:
# Techniques to reduce memory usage

random.seed(42)
wide_data = [
    (i, f"name_{i}", f"addr_{i}", f"email_{i}@test.com",
     random.choice(["A","B","C"]), random.random() * 1000,
     "x" * 200, "y" * 200, "z" * 200)
    for i in range(300_000)
]
wide_df = spark.createDataFrame(
    wide_data,
    ["id", "name", "address", "email", "group", "value", "col1", "col2", "col3"]
)

# Technique 1: select only needed columns first (Column Pruning)
start = time.time()
wide_df.groupBy("group").agg(F.avg("value")).collect()
all_col_time = time.time() - start

start = time.time()
wide_df.select("group", "value").groupBy("group").agg(F.avg("value")).collect()
pruned_time = time.time() - start

print("Technique 1: Column Pruning")
print(f"  All columns: {all_col_time:.3f}s")
print(f"  Only needed: {pruned_time:.3f}s")

# Technique 2: Appropriate data types
print("\nTechnique 2: Data type optimization")
print("  StringType  → variable length, high memory")
print("  IntegerType → 4 bytes fixed")
print("  ShortType   → 2 bytes fixed")
print("  FloatType   → 4 bytes (half of DoubleType's 8 bytes)")

# Technique 3: Early filtering
start = time.time()
wide_df.join(
    spark.createDataFrame([("A",)], ["group"]),
    "group"
).select("id", "value").count()
late_filter_time = time.time() - start

start = time.time()
wide_df.filter(F.col("group") == "A") \
    .select("id", "value").count()
early_filter_time = time.time() - start

print(f"\nTechnique 3: Early filtering")
print(f"  Filter via join:   {late_filter_time:.3f}s")
print(f"  Direct filter:     {early_filter_time:.3f}s")

Technique 1: Column Pruning
  All columns: 0.389s
  Only needed: 0.245s

Technique 2: Data type optimization
  StringType  → variable length, high memory
  IntegerType → 4 bytes fixed
  ShortType   → 2 bytes fixed
  FloatType   → 4 bytes (half of DoubleType's 8 bytes)

Technique 3: Early filtering
  Filter via join:   0.659s
  Direct filter:     0.174s


---
## 7. Real-World Tuning Checklist

In [15]:
print("""
╔══════════════════════════════════════════════════════════════╗
║                  Spark Tuning Checklist                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📊 Data Level                                             ║
║  □ select only needed columns (Column Pruning)             ║
║  □ Apply early filters (Predicate Pushdown)                ║
║  □ Use appropriate data types                              ║
║  □ Use Parquet/ORC format (columnar compression)           ║
║                                                            ║
║  🔄 Shuffle Level                                          ║
║  □ Tune spark.sql.shuffle.partitions                       ║
║  □ Use broadcast() for small tables                        ║
║  □ Check data skew and apply Salting/AQE                   ║
║  □ Reduce data with aggregation/filter before joining      ║
║                                                            ║
║  💾 Memory Level                                           ║
║  □ Right-size executor memory and cores                    ║
║  □ Cache only repeatedly used data, unpersist after use    ║
║  □ Minimize collect() / toPandas()                         ║
║  □ Use built-in functions or Pandas UDF instead of Python UDF║
║  □ Increase partition count when Spill occurs              ║
║                                                            ║
║  🔧 Config Level                                           ║
║  □ Enable AQE (Spark 3.x)                                  ║
║  □ Consider Dynamic Allocation                             ║
║  □ Use G1 GC                                               ║
║  □ Consider Kryo serialization                             ║
║                                                            ║
║  🔍 Monitoring                                             ║
║  □ Spark UI > SQL tab: check execution plan                ║
║  □ Spark UI > Stages: Spill and Shuffle sizes              ║
║  □ Spark UI > Executors: GC Time, memory usage             ║
║  □ Spark UI > Storage: cache size                          ║
║                                                            ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║                  Spark Tuning Checklist                    ║
╠══════════════════════════════════════════════════════════════╣
║                                                            ║
║  📊 Data Level                                             ║
║  □ select only needed columns (Column Pruning)             ║
║  □ Apply early filters (Predicate Pushdown)                ║
║  □ Use appropriate data types                              ║
║  □ Use Parquet/ORC format (columnar compression)           ║
║                                                            ║
║  🔄 Shuffle Level                                          ║
║  □ Tune spark.sql.shuffle.partitions                       ║
║  □ Use broadcast() for small tables                        ║
║  □ Check data skew and apply Salting/AQE                   ║
║  □ Reduce data with aggregation/filter before joining      ║
║                                                   

---
## 📝 Key Summary

| Concept | Description |
|------|------|
| **Unified Memory** | Storage + Execution share a fluid boundary |
| **Storage Memory** | Stores cache() and broadcast data |
| **Execution Memory** | Buffers for Shuffle, Sort, Join, Aggregation |
| **User Memory** | UDF, Python objects, user data structures |
| **Spill** | Temporary disk write when memory is insufficient |
| **GC** | Takes longer with larger heap and more objects |
| **Off-heap** | Memory outside JVM heap (memoryOverhead) |

### OOM Response Summary
| Location | Parameter | Root Cause |
|------|---------|----------|
| Driver | `driver.memory` | collect, broadcast, metadata |
| Executor Heap | `executor.memory` | large partitions, skew, UDF |
| Executor Off-heap | `memoryOverhead` | Python worker, native lib |

### Next Step (Step 6)
- Spark Structured Streaming
- Micro-batch processing
- Watermark & windows
- Streaming + batch integration

In [16]:
spark.stop()
print("SparkSession stopped")

SparkSession stopped
